# 1. Purpose and Safety Scope
Task 04A tests installation/loading and **one source image, one target age, one saved output per model**.
This is NOT an experiment. Outputs are NOT research results or predictions of true future appearance.
SAM (candidate M0) and FADING (candidate M1) remain unaccepted until real Colab smoke tests pass.
No training, M2 selection, evaluation or research figures are performed.


## 2. Prerequisites
Run `notebooks/01_setup.ipynb` first with `GITHUB_BRANCH = "task-04a-model-feasibility"`.
Task 03's real FG-NET raw images and processed metadata must already be on Drive.
Follow `docs/model_feasibility.md` for explicit external setup, checkpoints, licenses and isolated interpreters.
Generation requires a GPU. Never install old model requirements into the canonical runtime.
SAM computes at 1024px internally (256px saved); FADING uses two internal prompts (one saved 512px output).
These reviewed internal size exceptions are explicit in config. Both bridges use FP32 pending FP16 validation.


In [ ]:
from pathlib import Path
import sys
import json
import subprocess
from importlib.metadata import version
PROJECT_ROOT = Path('/content/face-age-progression')
DRIVE_ROOT = Path('/content/drive/MyDrive/AI6132')
if not (PROJECT_ROOT / 'configs/models.yaml').is_file() or not (DRIVE_ROOT / 'data').is_dir():
    raise RuntimeError('Run canonical setup first; mount Drive and checkout Task 04A.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.config import load_config
from src.models.feasibility import select_source, check_external
from src.models.sam_adapter import SAMAdapter
from src.models.fading_adapter import FADINGAdapter
CONFIG = load_config(PROJECT_ROOT / 'configs/models.yaml')
CANONICAL_VERSIONS = {name: version(name) for name in ('torch', 'torchvision')}
print('Canonical versions:', CANONICAL_VERSIONS)
RUN_SMOKE = False  # Set True only after the explicit dependency/checkpoint checks pass.
TRUST_SAM_CHECKPOINT = False  # Set True only after verifying the upstream pickle checkpoint.
FADING_GENDER = None  # Set 'female' or 'male' for the upstream prompt; never inferred from FG-NET.


## 3. Runtime / GPU Diagnostics
The assigned GPU may differ from a T4. The actual worker interpreter/package versions are also recorded.


In [ ]:
from src.utils.device import device_info
import torch
print(json.dumps(device_info(), indent=2))
print('Canonical CUDA build:', torch.version.cuda)
GPU_AVAILABLE = torch.cuda.is_available()
if GPU_AVAILABLE:
    gpu = torch.cuda.get_device_properties(0)
    print('GPU:', gpu.name, 'Total VRAM bytes:', gpu.total_memory)
else:
    print('GPU generation blocked: select a Colab GPU runtime. Local checks remain available.')


## 4. Persistent Paths
External source checkouts stay outside Git. Models, caches, outputs and smoke JSON stay on Drive.


In [ ]:
RAW_ROOT = DRIVE_ROOT / 'data/fgnet/raw'
METADATA_PATH = DRIVE_ROOT / 'data/fgnet/processed/metadata.csv'
OUTPUT_ROOT = Path(CONFIG['smoke']['output_root'])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for model in ('sam', 'fading'):
    print(model, {key: CONFIG[model][key] for key in ('external_repository', 'python', 'checkpoint', 'cache')})
print('Smoke outputs:', OUTPUT_ROOT)


## 5. Select One FG-NET Smoke-Test Image
Stable selection by subject ID, age, relative path from real processed metadata, restricted to children.
A default older target of 30 is a visual editing condition, never a claim about the subject's future.
Both candidates use the same raw source and target. The raw image is never copied into Git.


In [ ]:
SOURCE = select_source(METADATA_PATH, RAW_ROOT,
                       target_age=CONFIG['smoke']['target_age'],
                       max_source_age=CONFIG['smoke']['max_source_age'])
print(json.dumps(SOURCE, indent=2))


## 6. SAM Dependency / Checkpoint Check
Manual setup recipe: `docs/model_feasibility.md` (isolated venv, constrained canonical torch/torchvision).
The local check verifies source files, checkpoint/predictor presence and exact clean upstream revision.
It does not load weights, prove availability of download links or establish CUDA compatibility.


In [ ]:
SAM_CONFIG = dict(CONFIG['sam'], seed=CONFIG['smoke']['seed'], trusted_checkpoint=TRUST_SAM_CHECKPOINT)
SAM_READY = False
try:
    print(json.dumps(check_external('sam', SAM_CONFIG), indent=2))
    subprocess.run([SAM_CONFIG['python'], '-m', 'pip', 'check'], check=True)
    SAM_READY = True
except Exception as error:
    print('SAM preflight failed:', type(error).__name__, str(error))


## 7. SAM One-Image Smoke Test
Enable `RUN_SMOKE` and verified checkpoint trust above. The worker aligns the source with dlib,
loads the checkpoint, and generates one target-age image. Success is skipped only for matching metadata.
Any CUDA extension, alignment, loading or memory failure is logged independently on Drive.


In [ ]:
if RUN_SMOKE and GPU_AVAILABLE:
    try:
        result = SAMAdapter(SAM_CONFIG).generate(SOURCE['source_image'], SOURCE['target_age'],
                                                OUTPUT_ROOT / 'sam.png', source_age=SOURCE['source_age'])
        print(json.dumps(result, indent=2))
    except Exception as error:
        print('SAM smoke failed:', type(error).__name__, str(error))
else:
    print('SAM generation not run. Enable RUN_SMOKE in a GPU runtime after preflight.')


## 8. FADING Dependency / Checkpoint Check
Use the documented separate Python 3.10 environment and pinned community compatibility implementation.
Do not downgrade canonical torch. Specialized pipeline components must exist locally; no Hub downloads occur.
This recipe and modern diffusers compatibility are unverified until the real worker succeeds.
Set the researcher-supplied `FADING_GENDER` prompt above; FG-NET metadata does not provide it.


In [ ]:
FADING_CONFIG = dict(CONFIG['fading'], seed=CONFIG['smoke']['seed'], gender=FADING_GENDER)
FADING_READY = False
try:
    print(json.dumps(check_external('fading', FADING_CONFIG), indent=2))
    subprocess.run([FADING_CONFIG['python'], '-m', 'pip', 'check'], check=True)
    FADING_READY = True
except Exception as error:
    print('FADING preflight failed:', type(error).__name__, str(error))


## 9. FADING One-Image Smoke Test
Run after SAM exits. Null-text inversion optimizes only per-image embeddings; it requires gradients.
Upstream model specialization is not trained. Only one target age and one saved output are requested.
Peak GPU allocated/reserved memory and cold smoke duration include model loading, inversion and saving.


In [ ]:
if RUN_SMOKE and GPU_AVAILABLE:
    try:
        result = FADINGAdapter(FADING_CONFIG).generate(SOURCE['source_image'], SOURCE['source_age'],
                                                       SOURCE['target_age'], OUTPUT_ROOT / 'fading.png')
        print(json.dumps(result, indent=2))
    except Exception as error:
        print('FADING smoke failed:', type(error).__name__, str(error))
else:
    print('FADING generation not run. Enable RUN_SMOKE in a GPU runtime after preflight.')


## 10. Feasibility Summary
Read actual persisted statuses and measurements. Missing/pending/failed records do not establish feasibility.
Inspect the generated PNGs manually. No identity, age accuracy, image-quality or research-result claim is made.
Save dependency installation logs on Drive too. See documentation for timing scope and restart limitations.


In [ ]:
for model in ('sam', 'fading'):
    path = OUTPUT_ROOT / f'{model}.json'
    if not path.is_file():
        print(model, 'No smoke metadata; real GPU feasibility pending.')
        continue
    record = json.loads(path.read_text())
    keys = ('status', 'source_image', 'source_age', 'target_age', 'precision', 'device',
            'output_path', 'output_resolution', 'wall_clock_inference_seconds',
            'peak_cuda_memory_bytes', 'peak_cuda_reserved_bytes', 'upstream', 'versions', 'error_message')
    print(model, json.dumps({key: record.get(key) for key in keys}, indent=2))
assert CANONICAL_VERSIONS == {name: version(name) for name in CANONICAL_VERSIONS}, 'Canonical builds changed!'
print('CPU tests alone do not establish real Colab GPU feasibility. Neither candidate is automatically accepted.')
